# Modul 17: PyTorch-Tensoren, DataLoader und dichte Netze

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Tensoren und Loader, Dichtes PyTorch-Netz  
    **Erwarteter Schwierigkeitsgrad:** Mittlere bis fortgeschrittene PyTorch-Anwendung  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie untersuchen Tensoren und Autograd, organisieren Daten mit TensorDataset und DataLoader und schreiben anschließend ein vollständiges Training für ein kleines dichtes PyTorch-Netz. Metriken, Eval-Modus und reproduzierbares Speichern schließen den Workflow ab.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_17A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_17B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - PyTorch-Tensoren, Formen, Datentypen, NumPy-Konvertierung und Broadcasting sicher verwenden.
- Automatische Gradienten mit Autograd berechnen und kontrollieren.
- Trainings-, Validierungs- und Testdaten mit TensorDataset und DataLoader organisieren.
- Ein eigenes nn.Module mit einer korrekten forward-Methode definieren.
- Loss, Optimierer sowie train- und eval-Modus in einer Trainingsschleife korrekt einsetzen.
- Metriken visualisieren und Modellgewichte reproduzierbar speichern und laden.

    ## Bewertete Fähigkeiten

    - torch.Tensor, dtype, device, NumPy-Konvertierung und Broadcasting
- requires_grad, backward und grad
- TensorDataset und DataLoader
- nn.Module, forward und BCEWithLogitsLoss
- Training, Validierung, eval, no_grad und state_dict

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames PyTorch-Setup
import io
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein nichtlinearer Zweiklassendatensatz wird lokal erzeugt.
X_17, y_17 = make_moons(n_samples=720, noise=0.24, random_state=RANDOM_SEED)
X_train_valid_17, X_test_17, y_train_valid_17, y_test_17 = train_test_split(
    X_17,
    y_17,
    test_size=0.20,
    stratify=y_17,
    random_state=RANDOM_SEED,
)
X_train_17, X_valid_17, y_train_17, y_valid_17 = train_test_split(
    X_train_valid_17,
    y_train_valid_17,
    test_size=0.25,
    stratify=y_train_valid_17,
    random_state=RANDOM_SEED,
)

scaler_17 = StandardScaler()
X_train_17 = scaler_17.fit_transform(X_train_17).astype("float32")
X_valid_17 = scaler_17.transform(X_valid_17).astype("float32")
X_test_17 = scaler_17.transform(X_test_17).astype("float32")
y_train_17 = y_train_17.astype("float32")
y_valid_17 = y_valid_17.astype("float32")
y_test_17 = y_test_17.astype("float32")

print("Train/Valid/Test:", X_train_17.shape, X_valid_17.shape, X_test_17.shape)

print("PyTorch-Version:", torch.__version__)
print("Gerät:", DEVICE)


## Aufgabe 1: Tensoren, Broadcasting und Autograd

    Untersuchen Sie grundlegende PyTorch-Tensoroperationen.

1. Erzeugen Sie einen Skalar, Vektor, eine Matrix und einen kleinen Bildstapel mit ausdrücklich gewählten Datentypen.
2. Geben Sie Form, Dimensionen, Datentyp und Gerät aus.
3. Konvertieren Sie eine NumPy-Matrix zu PyTorch und wieder zurück. Vermeiden Sie eine unbeabsichtigte gemeinsame Speicheränderung.
4. Standardisieren Sie eine `3 x 2`-Matrix durch Broadcasting.
5. Berechnen Sie für `loss(w) = mean((w*x-y)^2)` den Gradienten mit `backward()` und vergleichen Sie ihn mit der manuellen Ableitung.

> **Hinweis:** Prüfen Sie bei jedem Tensor sowohl Form als auch Datentyp und Gerät.

In [ ]:
feature_matrix_17 = torch.tensor(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=torch.float32,
)
means_17 = torch.tensor([4.0, 14.0], dtype=torch.float32)
stds_17 = torch.tensor([2.0, 4.0], dtype=torch.float32)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Tensoren, Broadcasting und Autograd
#
# Ziel dieser Codezelle:
# Untersuchen Sie grundlegende PyTorch-Tensoroperationen. 1. Erzeugen Sie einen
# Skalar, Vektor, eine Matrix und einen kleinen Bildstapel mit ausdrücklich
# gewählten Datentypen. 2. Geben Sie Form, Dimensionen, Datentyp un...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

feature_matrix_17 = torch.tensor(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=torch.float32,
)
means_17 = torch.tensor([4.0, 14.0], dtype=torch.float32)
stds_17 = torch.tensor([2.0, 4.0], dtype=torch.float32)

scalar_17 = torch.tensor(3.5, dtype=torch.float32)
vector_17 = torch.tensor([1, 2, 3], dtype=torch.int64)
matrix_17 = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=torch.float32)
image_batch_17 = torch.zeros((4, 1, 8, 8), dtype=torch.float32)

for name, tensor in {
    "Skalar": scalar_17,
    "Vektor": vector_17,
    "Matrix": matrix_17,
    "Bildstapel": image_batch_17,
}.items():
    print(
        name,
        "Form:", tuple(tensor.shape),
        "Dimensionen:", tensor.ndim,
        "dtype:", tensor.dtype,
        "Gerät:", tensor.device,
    )

# torch.from_numpy kann Speicher mit NumPy teilen. copy() vor der
# Konvertierung und clone() beim Rückweg machen die Unabhängigkeit klar.
numpy_source_17 = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
tensor_from_numpy_17 = torch.from_numpy(numpy_source_17.copy())
numpy_roundtrip_17 = tensor_from_numpy_17.detach().cpu().numpy().copy()
numpy_source_17[0, 0] = 999.0
assert numpy_roundtrip_17[0, 0] == 1.0

# Der Vektor der Länge zwei wird auf jede Zeile der Matrix angewendet.
standardized_17 = (feature_matrix_17 - means_17) / stds_17
expected_17 = torch.tensor([[-1.0, -1.0], [0.0, 0.0], [1.0, 1.0]])
torch.testing.assert_close(standardized_17, expected_17)
print("Standardisiert:\n", standardized_17)

x_autograd_17 = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
y_autograd_17 = torch.tensor([2.0, 4.0, 5.0], dtype=torch.float32)
w_autograd_17 = torch.tensor(1.5, dtype=torch.float32, requires_grad=True)

predictions_17 = w_autograd_17 * x_autograd_17
loss_17 = torch.mean((predictions_17 - y_autograd_17) ** 2)
loss_17.backward()
automatic_gradient_17 = w_autograd_17.grad.detach().clone()

manual_gradient_17 = torch.mean(
    2.0 * (w_autograd_17.detach() * x_autograd_17 - y_autograd_17) * x_autograd_17
)
torch.testing.assert_close(automatic_gradient_17, manual_gradient_17)

print("Verlust:", round(float(loss_17.item()), 5))
print("Autograd-Gradient:", round(float(automatic_gradient_17.item()), 5))
print("Manueller Gradient:", round(float(manual_gradient_17.item()), 5))

### Reflexion zu Aufgabe 1

PyTorch verwendet für Bilder häufig das Format `(Batch, Kanäle, Höhe, Breite)`, im Gegensatz zum standardmäßigen channels-last-Format von Keras. Tensoren können Speicher mit NumPy teilen, weshalb eine bewusste Kopie wichtig sein kann. Autograd speichert den Rechengraphen dynamisch und sammelt Gradienten standardmäßig in `.grad`. Vor einem neuen Optimierungsschritt müssen diese Gradienten daher typischerweise zurückgesetzt werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: TensorDataset und DataLoader korrekt aufteilen

    Organisieren Sie die vorbereiteten Daten für das Training.

1. Wandeln Sie Merkmale und Labels in `float32`-Tensoren um. Labels sollen die Form `(N, 1)` besitzen.
2. Erstellen Sie getrennte `TensorDataset`-Objekte für Training, Validierung und Test.
3. Erstellen Sie DataLoader mit Batchgröße 32. Mischen Sie nur das Training.
4. Verwenden Sie einen festen Generator für reproduzierbares Shuffling.
5. Untersuchen Sie einen Batch und prüfen Sie Formen, Datentypen und Beispielzahl.
6. Erklären Sie, warum der Test-Loader nicht zum Modell- oder Schwellenwertvergleich verwendet werden darf.

> **Hinweis:** Richten Sie die Labelform bereits im Dataset an der späteren Loss-Funktion aus.

In [ ]:
batch_size_17 = 32

# Speichern Sie die Loader als train_loader_17, valid_loader_17 und
# test_loader_17, damit spätere Aufgaben sie verwenden können.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: TensorDataset und DataLoader korrekt aufteilen
#
# Ziel dieser Codezelle:
# Organisieren Sie die vorbereiteten Daten für das Training. 1. Wandeln Sie Merkmale
# und Labels in float32-Tensoren um. Labels sollen die Form (N, 1) besitzen. 2.
# Erstellen Sie getrennte TensorDataset-Objekte für Traini...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

batch_size_17 = 32

X_train_tensor_17 = torch.tensor(X_train_17, dtype=torch.float32)
X_valid_tensor_17 = torch.tensor(X_valid_17, dtype=torch.float32)
X_test_tensor_17 = torch.tensor(X_test_17, dtype=torch.float32)

# BCEWithLogitsLoss erwartet hier eine Float-Zielmatrix mit derselben
# Form wie die Logits des Modells.
y_train_tensor_17 = torch.tensor(y_train_17, dtype=torch.float32).reshape(-1, 1)
y_valid_tensor_17 = torch.tensor(y_valid_17, dtype=torch.float32).reshape(-1, 1)
y_test_tensor_17 = torch.tensor(y_test_17, dtype=torch.float32).reshape(-1, 1)

train_dataset_17 = TensorDataset(X_train_tensor_17, y_train_tensor_17)
valid_dataset_17 = TensorDataset(X_valid_tensor_17, y_valid_tensor_17)
test_dataset_17 = TensorDataset(X_test_tensor_17, y_test_tensor_17)

# Der Generator kontrolliert die Reihenfolge der gemischten
# Trainingsbeispiele unabhängig vom globalen Zustand.
loader_generator_17 = torch.Generator().manual_seed(RANDOM_SEED)
train_loader_17 = DataLoader(
    train_dataset_17,
    batch_size=batch_size_17,
    shuffle=True,
    generator=loader_generator_17,
    num_workers=0,
)
valid_loader_17 = DataLoader(
    valid_dataset_17,
    batch_size=batch_size_17,
    shuffle=False,
    num_workers=0,
)
test_loader_17 = DataLoader(
    test_dataset_17,
    batch_size=batch_size_17,
    shuffle=False,
    num_workers=0,
)

first_features_17, first_labels_17 = next(iter(train_loader_17))
assert first_features_17.shape == (batch_size_17, 2)
assert first_labels_17.shape == (batch_size_17, 1)
assert first_features_17.dtype == torch.float32
assert first_labels_17.dtype == torch.float32
assert len(train_dataset_17) + len(valid_dataset_17) + len(test_dataset_17) == len(X_17)

print("Trainingsbeispiele:", len(train_dataset_17))
print("Validierungsbeispiele:", len(valid_dataset_17))
print("Testbeispiele:", len(test_dataset_17))
print("Erster Merkmalsbatch:", tuple(first_features_17.shape))
print("Erster Labelbatch:", tuple(first_labels_17.shape))

### Reflexion zu Aufgabe 2

Der DataLoader erstellt Batches und kann die Trainingsreihenfolge pro Epoche mischen. Validierungs- und Testdaten benötigen kein Shuffling, weil keine Parameterupdates stattfinden. Das Testset bleibt bis nach allen Entscheidungen zurückgehalten. Würde es zur Architektur-, Epochen- oder Schwellenwertwahl verwendet, wäre die abschließende Testleistung optimistisch verzerrt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Ein eigenes nn.Module definieren

    Implementieren Sie ein dichtes binäres Netz als eigene PyTorch-Klasse.

1. Erben Sie von `nn.Module`.
2. Definieren Sie eine Architektur `2 -> 24 -> 12 -> 1` mit ReLU in den verborgenen Schichten.
3. Geben Sie in `forward` **Logits** und keine Sigmoid-Wahrscheinlichkeiten zurück.
4. Verschieben Sie das Modell auf `DEVICE`.
5. Richten Sie `BCEWithLogitsLoss` und Adam ein.
6. Prüfen Sie die Ausgabeform eines Batches und zählen Sie trainierbare Parameter.

> **Hinweis:** Kombinieren Sie Sigmoid nicht zusätzlich mit `BCEWithLogitsLoss`.

In [ ]:
# Speichern Sie Modell, Loss und Optimierer als model_17,
# criterion_17 und optimizer_17.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Ein eigenes nn.Module definieren
#
# Ziel dieser Codezelle:
# Implementieren Sie ein dichtes binäres Netz als eigene PyTorch-Klasse. 1. Erben
# Sie von nn.Module. 2. Definieren Sie eine Architektur 2 - 24 - 12 - 1 mit ReLU in
# den verborgenen Schichten. 3. Geben Sie in forward Logi...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

class DenseBinaryNet17(nn.Module):
    def __init__(self, input_features=2):
        super().__init__()
        # Sequential ist für eine einfache Feedforward-Struktur klar
        # lesbar, bleibt aber vollständig Teil des eigenen nn.Module.
        self.network = nn.Sequential(
            nn.Linear(input_features, 24),
            nn.ReLU(),
            nn.Linear(24, 12),
            nn.ReLU(),
            nn.Linear(12, 1),
        )

    def forward(self, features):
        # Die letzte lineare Schicht liefert rohe Logits. Sigmoid wird
        # absichtlich nicht hier aufgerufen, weil BCEWithLogitsLoss
        # Sigmoid und Kreuzentropie numerisch stabil kombiniert.
        return self.network(features)

torch.manual_seed(RANDOM_SEED)
model_17 = DenseBinaryNet17(input_features=2).to(DEVICE)
criterion_17 = nn.BCEWithLogitsLoss()
optimizer_17 = torch.optim.Adam(model_17.parameters(), lr=0.003)

example_features_17, _ = next(iter(train_loader_17))
example_logits_17 = model_17(example_features_17.to(DEVICE))
trainable_parameters_17 = sum(
    parameter.numel()
    for parameter in model_17.parameters()
    if parameter.requires_grad
)

assert example_logits_17.shape == (batch_size_17, 1)
print(model_17)
print("Ausgabeform:", tuple(example_logits_17.shape))
print("Trainierbare Parameter:", trainable_parameters_17)

### Reflexion zu Aufgabe 3

`nn.Module` registriert Untermodule und Parameter automatisch, wenn sie als Attribute angelegt werden. Die `forward`-Methode beschreibt den Datenfluss. `BCEWithLogitsLoss` ist gegenüber einer getrennten Sigmoid- und Logarithmusberechnung numerisch stabiler. Wahrscheinlichkeiten werden erst bei der Interpretation mit `torch.sigmoid(logits)` berechnet.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Trainings- und Evaluationsschleifen schreiben

    Trainieren Sie `model_17` mit klar getrennten Trainings- und Evaluationsphasen.

1. Schreiben Sie eine Funktion für eine Trainingsepoche mit `model.train()`, `zero_grad()`, Vorwärtslauf, Loss, `backward()` und `step()`.
2. Schreiben Sie eine Evaluationsfunktion mit `model.eval()` und `torch.no_grad()`.
3. Speichern Sie pro Epoche Loss und Accuracy für Training und Validierung.
4. Implementieren Sie einfaches Early Stopping anhand des Validierungsverlusts und sichern Sie die besten Gewichte im Speicher.
5. Stellen Sie die besten Gewichte wieder her und bewerten Sie das Testset genau einmal.

> **Hinweis:** Multiplizieren Sie den Batch-Loss mit der Batchgröße, bevor Sie über unterschiedlich große Batches mitteln.

In [ ]:
max_epochs_17 = 10 if FAST_MODE else 80
patience_17 = 10

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Trainings- und Evaluationsschleifen schreiben
#
# Ziel dieser Codezelle:
# Trainieren Sie model17 mit klar getrennten Trainings- und Evaluationsphasen. 1.
# Schreiben Sie eine Funktion für eine Trainingsepoche mit model.train(),
# zerograd(), Vorwärtslauf, Loss, backward() und step(). 2. Schreib...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

max_epochs_17 = 10 if FAST_MODE else 80
patience_17 = 10

def run_training_epoch_17(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for features, labels in loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        # Gradienten werden in PyTorch akkumuliert. Daher müssen sie
        # vor jedem neuen Batch ausdrücklich auf null gesetzt werden.
        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = features.shape[0]
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).to(labels.dtype)
        total_loss += float(loss.item()) * batch_size
        total_correct += int((predictions == labels).sum().item())
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

def evaluate_17(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probabilities = []
    all_labels = []

    # no_grad verhindert den Aufbau eines Rechengraphen und spart
    # Speicher bei Validierung und Test.
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(features)
            loss = criterion(logits, labels)

            total_loss += float(loss.item()) * features.shape[0]
            all_probabilities.append(torch.sigmoid(logits).cpu())
            all_labels.append(labels.cpu())

    probabilities = torch.cat(all_probabilities).numpy().ravel()
    labels = torch.cat(all_labels).numpy().ravel().astype(int)
    predictions = (probabilities >= 0.5).astype(int)
    mean_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(labels, predictions)
    return mean_loss, accuracy, probabilities, labels

history_17 = {
    "train_loss": [],
    "train_accuracy": [],
    "valid_loss": [],
    "valid_accuracy": [],
}
best_valid_loss_17 = np.inf
best_state_17 = None
epochs_without_improvement_17 = 0

for epoch in range(max_epochs_17):
    train_loss_17, train_accuracy_17 = run_training_epoch_17(
        model_17,
        train_loader_17,
        criterion_17,
        optimizer_17,
    )
    valid_loss_17, valid_accuracy_17, _, _ = evaluate_17(
        model_17,
        valid_loader_17,
        criterion_17,
    )

    history_17["train_loss"].append(train_loss_17)
    history_17["train_accuracy"].append(train_accuracy_17)
    history_17["valid_loss"].append(valid_loss_17)
    history_17["valid_accuracy"].append(valid_accuracy_17)

    if valid_loss_17 < best_valid_loss_17 - 1e-4:
        best_valid_loss_17 = valid_loss_17
        # CPU-Kopien sind unabhängig vom aktuellen Gerät und vom
        # weiteren Training des Modells.
        best_state_17 = {
            name: value.detach().cpu().clone()
            for name, value in model_17.state_dict().items()
        }
        epochs_without_improvement_17 = 0
    else:
        epochs_without_improvement_17 += 1

    if epochs_without_improvement_17 >= patience_17:
        break

model_17.load_state_dict(best_state_17)
model_17.to(DEVICE)
test_loss_17, test_accuracy_17, test_probabilities_17, test_labels_17 = evaluate_17(
    model_17,
    test_loader_17,
    criterion_17,
)
test_predictions_17 = (test_probabilities_17 >= 0.5).astype(int)
test_balanced_17 = balanced_accuracy_score(test_labels_17, test_predictions_17)

print("Trainierte Epochen:", len(history_17["train_loss"]))
print("Bester Validierungsverlust:", round(float(best_valid_loss_17), 4))
print("Testverlust:", round(float(test_loss_17), 4))
print("Testgenauigkeit:", round(float(test_accuracy_17), 4))
print("Test Balanced Accuracy:", round(float(test_balanced_17), 4))

### Reflexion zu Aufgabe 4

`model.train()` und `model.eval()` steuern das Verhalten zustandsabhängiger Schichten wie Dropout und BatchNorm. `torch.no_grad()` ist davon getrennt und deaktiviert die Gradientenaufzeichnung. Beide sind für eine korrekte und effiziente Bewertung wichtig. Early Stopping darf nur Validierungsdaten beobachten. Die Testdaten werden erst nach Wiederherstellung des besten Zustands verwendet.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: Metriken visualisieren und Gewichte sicher laden

    Schließen Sie das Mini-Projekt mit Diagnose und Reproduzierbarkeit ab.

1. Stellen Sie Loss und Accuracy für Training und Validierung dar.
2. Erstellen Sie eine Konfusionsmatrix für das Testset und zeigen Sie fünf besonders unsichere Beispiele.
3. Speichern Sie `state_dict`, Architekturmetadaten, Schwellenwert und Seed in einem In-Memory-Artefakt.
4. Erzeugen Sie eine neue Modellinstanz, laden Sie den Zustand mit `map_location` und versetzen Sie sie in den Eval-Modus.
5. Bestätigen Sie, dass die neue Instanz für zwölf Referenzbeispiele dieselben Logits und Klassen liefert.
6. Erläutern Sie, warum ein `state_dict` ohne Architektur- und Vorverarbeitungsinformationen kein vollständiges Produktionsartefakt ist.

> **Hinweis:** Erstellen Sie die neue Modellinstanz mit exakt derselben Architektur, bevor Sie den Zustand laden.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: Metriken visualisieren und Gewichte sicher laden
#
# Ziel dieser Codezelle:
# Schließen Sie das Mini-Projekt mit Diagnose und Reproduzierbarkeit ab. 1. Stellen
# Sie Loss und Accuracy für Training und Validierung dar. 2. Erstellen Sie eine
# Konfusionsmatrix für das Testset und zeigen Sie fünf beso...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

history_frame_17 = pd.DataFrame(history_17)

for metric_name, label in [("loss", "Loss"), ("accuracy", "Accuracy")]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_frame_17[f"train_{metric_name}"], label="Training")
    ax.plot(history_frame_17[f"valid_{metric_name}"], label="Validierung")
    ax.set_title(f"PyTorch-Training: {label}")
    ax.set_xlabel("Epoche")
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

confusion_17 = confusion_matrix(test_labels_17, test_predictions_17)
print("Konfusionsmatrix:\n", confusion_17)

# Unsicherheit wird als Abstand zur Entscheidungsschwelle 0.5 gemessen.
uncertain_indices_17 = np.argsort(np.abs(test_probabilities_17 - 0.5))[:5]
uncertain_table_17 = pd.DataFrame(
    {
        "probability_class_1": test_probabilities_17[uncertain_indices_17],
        "prediction": test_predictions_17[uncertain_indices_17],
        "true_label": test_labels_17[uncertain_indices_17],
        "feature_1": X_test_17[uncertain_indices_17, 0],
        "feature_2": X_test_17[uncertain_indices_17, 1],
    }
)
print("Unsichere Testbeispiele:")
print(uncertain_table_17.round(3).to_string(index=False))

reference_features_17 = torch.tensor(
    X_test_17[:12],
    dtype=torch.float32,
    device=DEVICE,
)
model_17.eval()
with torch.no_grad():
    reference_logits_17 = model_17(reference_features_17).cpu()

artifact_17 = {
    "model_state_dict": {
        name: value.detach().cpu()
        for name, value in model_17.state_dict().items()
    },
    "metadata": {
        "architecture": "2-24-12-1 ReLU MLP",
        "input_features": 2,
        "threshold": 0.5,
        "random_seed": RANDOM_SEED,
        "scaler_mean": scaler_17.mean_.tolist(),
        "scaler_scale": scaler_17.scale_.tolist(),
    },
}

# BytesIO simuliert eine Datei, ohne einen lokalen Pfad vorauszusetzen.
buffer_17 = io.BytesIO()
torch.save(artifact_17, buffer_17)
buffer_17.seek(0)
try:
    loaded_artifact_17 = torch.load(
        buffer_17,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    # Ältere PyTorch-Versionen kennen weights_only noch nicht.
    buffer_17.seek(0)
    loaded_artifact_17 = torch.load(buffer_17, map_location=DEVICE)

loaded_model_17 = DenseBinaryNet17(
    input_features=loaded_artifact_17["metadata"]["input_features"]
).to(DEVICE)
loaded_model_17.load_state_dict(loaded_artifact_17["model_state_dict"])
loaded_model_17.eval()

with torch.no_grad():
    loaded_logits_17 = loaded_model_17(reference_features_17).cpu()
torch.testing.assert_close(reference_logits_17, loaded_logits_17)

threshold_17 = loaded_artifact_17["metadata"]["threshold"]
original_classes_17 = (torch.sigmoid(reference_logits_17) >= threshold_17).int()
loaded_classes_17 = (torch.sigmoid(loaded_logits_17) >= threshold_17).int()
torch.testing.assert_close(original_classes_17, loaded_classes_17)

print("Artefaktmetadaten:", loaded_artifact_17["metadata"])
print("Ladeprüfung bestanden: Logits und Klassen sind identisch.")

### Reflexion zu Aufgabe 5

Ein `state_dict` enthält Parameterwerte, aber nicht automatisch die Python-Klasse, Eingabespalten, Skalierungsparameter, Labelbedeutung oder Entscheidungsschwelle. Ohne diese Informationen kann ein technisch ladbares Modell fachlich falsch verwendet werden. Ein belastbares Artefakt benötigt deshalb Architekturcode oder eine klar versionierte Definition, Vorverarbeitungsmetadaten, Abhängigkeiten, Referenztests und eine dokumentierte Modellkarte.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.